In [4]:
from pathlib import Path
import pandas as pd
import json

In [5]:
# Diretório atual: out/agg
BASE_DIR = Path.cwd()

# Arquivo CSV original
PR_FILE = BASE_DIR / "pr_slim.csv"

# Diretório onde serão salvos os JSONs
OUTPUT_DIR = BASE_DIR / "public" / "data"

# Cria a pasta caso ainda não exista
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório atual:", BASE_DIR.resolve())
print("Arquivo de entrada:", PR_FILE.resolve())
print("Diretório de saída:", OUTPUT_DIR.resolve())

if not PR_FILE.exists():
    raise FileNotFoundError(
        f"Não foi encontrado o arquivo: {PR_FILE}"
    )

print("\nCaminhos configurados corretamente!")

Diretório atual: C:\Users\fpaul\Documents\GitHub\dataviz-final-work\out\agg
Arquivo de entrada: C:\Users\fpaul\Documents\GitHub\dataviz-final-work\out\agg\pr_slim.csv
Diretório de saída: C:\Users\fpaul\Documents\GitHub\dataviz-final-work\out\agg\public\data

Caminhos configurados corretamente!


In [6]:
# Carrega o CSV
pr = pd.read_csv(PR_FILE, low_memory=False)

# Colunas necessárias
required_columns = {
    "agent",
    "review_type",
    "date",
    "dow",
    "hour"
}

# Verifica se todas as colunas existem
missing_columns = required_columns - set(pr.columns)

if missing_columns:
    raise ValueError(
        f"Colunas ausentes no pr_slim.csv: {sorted(missing_columns)}"
    )

print("Dados carregados com sucesso!")
print(f"Total de Pull Requests: {len(pr):,}")
print(f"Quantidade de agentes: {pr['agent'].nunique()}")
print(f"Tipos de revisão: {pr['review_type'].nunique()}")

print("\nAgentes encontrados:")
print(sorted(pr["agent"].dropna().unique()))

print("\nTipos de revisão encontrados:")
print(sorted(pr["review_type"].dropna().unique()))

Dados carregados com sucesso!
Total de Pull Requests: 71,677
Quantidade de agentes: 6
Tipos de revisão: 4

Agentes encontrados:
['Claude Code', 'Codex', 'Copilot', 'Cursor', 'Devin', 'Jules']

Tipos de revisão encontrados:
['bot_only', 'human', 'human_changes', 'none']


In [7]:
# Seleciona as colunas necessárias
review = pr[["agent", "review_type"]].copy()

# Remove registros sem identificação de agente
review = review.dropna(subset=["agent"])

# Classifica valores ausentes
review["review_type"] = (
    review["review_type"]
    .astype("object")
    .fillna("unknown")
    .astype(str)
)

# Conta os PRs por agente e tipo de revisão
review_counts = (
    review.groupby(
        ["agent", "review_type"],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# Total de PRs por agente
review_counts["total_agent"] = (
    review_counts
    .groupby("agent")["count"]
    .transform("sum")
)

# Percentual de cada tipo de revisão dentro do agente
review_counts["percentage"] = (
    review_counts["count"] /
    review_counts["total_agent"] * 100
)

# Organiza os registros
review_counts = review_counts.sort_values(
    ["agent", "review_type"]
).reset_index(drop=True)

print("Distribuição dos tipos de revisão:")
display(review_counts)

Distribuição dos tipos de revisão:


,agent,review_type,count,total_agent,percentage
0,Claude Code,bot_only,379,1942,19.515963
1,Claude Code,human,547,1942,28.166838
2,Claude Code,human_changes,58,1942,2.986612
3,Claude Code,none,958,1942,49.330587
4,Codex,bot_only,2469,35521,6.950818
5,Codex,human,1898,35521,5.343318
6,Codex,human_changes,102,35521,0.287154
7,Codex,none,31052,35521,87.418710
8,Copilot,bot_only,839,23496,3.570821
9,Copilot,human,9076,23496,38.627852


In [8]:
# Seleciona as colunas necessárias
hourly = pr[
    ["date", "dow", "hour", "agent"]
].copy()

# Converte os tipos
hourly["date"] = pd.to_datetime(
    hourly["date"],
    errors="coerce"
)

hourly["dow"] = pd.to_numeric(
    hourly["dow"],
    errors="coerce"
)

hourly["hour"] = pd.to_numeric(
    hourly["hour"],
    errors="coerce"
)

# Corrige dias da semana inválidos utilizando a própria data
# Convenção: 0 = segunda-feira e 6 = domingo
invalid_dow = ~hourly["dow"].between(0, 6)

hourly.loc[invalid_dow, "dow"] = (
    hourly.loc[invalid_dow, "date"].dt.dayofweek
)

# Remove registros inválidos
hourly = hourly.dropna(
    subset=["date", "dow", "hour", "agent"]
).copy()

# Converte para inteiros
hourly["dow"] = hourly["dow"].astype(int)
hourly["hour"] = hourly["hour"].astype(int)

# Mantém somente horários válidos
hourly = hourly[
    hourly["hour"].between(0, 23)
].copy()

# Padroniza a data
hourly["date"] = hourly["date"].dt.strftime("%Y-%m-%d")

# Agrupa por data, dia da semana, hora e agente
hourly_counts = (
    hourly.groupby(
        ["date", "dow", "hour", "agent"],
        observed=True
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        ["date", "dow", "hour", "agent"]
    )
    .reset_index(drop=True)
)

print("Atividade por dia e horário:")
display(hourly_counts.head(20))

print(f"\nTotal de combinações agregadas: {len(hourly_counts):,}")

Atividade por dia e horário:


,date,dow,hour,agent,count
0,2024-12-24,1,0,Devin,1
1,2024-12-24,1,3,Devin,3
2,2024-12-24,1,5,Devin,1
3,2024-12-24,1,11,Devin,1
4,2024-12-24,1,13,Devin,1
5,2024-12-24,1,16,Devin,1
6,2024-12-24,1,17,Devin,1
7,2024-12-24,1,18,Devin,1
8,2024-12-24,1,19,Devin,1
9,2024-12-25,2,0,Devin,1



Total de combinações agregadas: 14,225


In [9]:
def write_json(dataframe, filename):
    """
    Salva um DataFrame em formato JSON.
    """

    output_file = OUTPUT_DIR / filename

    # Converte o DataFrame para registros JSON serializáveis
    data = json.loads(
        dataframe.to_json(
            orient="records",
            force_ascii=False
        )
    )

    # Grava o arquivo
    with output_file.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        )

    print(f"Arquivo gerado: {output_file.name}")
    print(f"Tamanho: {output_file.stat().st_size / 1024:.2f} KB")


# Gera os dois arquivos
write_json(
    review_counts,
    "review_types.json"
)

write_json(
    hourly_counts,
    "hourly_activity.json"
)

print("\nPreparação dos dados concluída!")

Arquivo gerado: review_types.json
Tamanho: 3.40 KB
Arquivo gerado: hourly_activity.json
Tamanho: 1527.47 KB

Preparação dos dados concluída!


In [10]:
expected_files = [
    "review_types.json",
    "hourly_activity.json"
]

print("=" * 55)
print("VERIFICAÇÃO DOS ARQUIVOS")
print("=" * 55)

for filename in expected_files:

    path = OUTPUT_DIR / filename

    if not path.exists():
        print(f"ERRO: {filename} não foi encontrado!")
        continue

    # Confere se o JSON pode ser lido
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    print(f"\n{filename}")
    print(f"Registros: {len(data):,}")
    print(f"Tamanho: {path.stat().st_size / 1024:.2f} KB")
    print("JSON válido: Sim")

    if data:
        print("Primeiro registro:", data[0])

print("\nDiretório de saída:")
print(OUTPUT_DIR.resolve())

VERIFICAÇÃO DOS ARQUIVOS

review_types.json
Registros: 24
Tamanho: 3.40 KB
JSON válido: Sim
Primeiro registro: {'agent': 'Claude Code', 'review_type': 'bot_only', 'count': 379, 'total_agent': 1942, 'percentage': 19.5159629248}

hourly_activity.json
Registros: 14,225
Tamanho: 1527.47 KB
JSON válido: Sim
Primeiro registro: {'date': '2024-12-24', 'dow': 1, 'hour': 0, 'agent': 'Devin', 'count': 1}

Diretório de saída:
C:\Users\fpaul\Documents\GitHub\dataviz-final-work\out\agg\public\data
